# 🇮🇩➡️🇩🇪 Machine Translator: Indonesia → Jerman

**Dibuat menggunakan:**
- 🤗 `Helsinki-NLP/opus-mt-id-de` — Model pre-trained khusus Bahasa Indonesia → Jerman
- 🎨 `Gradio` — UI web interaktif langsung dari Colab
- 🔁 `transformers` — Library HuggingFace untuk load & run model

---

## 📋 Alur Kerja Notebook Ini

| No | Cell | Fungsi |
|----|------|--------|
| 1 | Install Library | Instal semua dependensi yang dibutuhkan |
| 2 | Import & Load Model | Unduh model Helsinki-NLP dari HuggingFace Hub |
| 3 | Fungsi Translate | Logika inti penerjemahan |
| 4 | Uji Coba Cepat | Test terjemahan tanpa UI |
| 5 | Bangun UI Gradio | Tampilan web interaktif |
| 6 | Jalankan Aplikasi | Launch & buka link publik |

> ⚠️ **Catatan:** Model akan diunduh otomatis saat pertama kali dijalankan (~300MB). Pastikan koneksi internet stabil.

---
## 📦 CELL 1 — Install Library yang Dibutuhkan

Jalankan cell ini **sekali saja** di awal. Proses install biasanya 1–2 menit.

- `transformers` → untuk load model NLP dari HuggingFace
- `sentencepiece` → tokenizer yang dipakai model Helsinki-NLP
- `torch` → backend deep learning (biasanya sudah ada di Colab)
- `gradio` → membuat tampilan UI web

In [1]:
# ============================================================
#  CELL 1 — Install semua library yang dibutuhkan
# ============================================================

!pip install transformers sentencepiece torch gradio --quiet

print("✅ Semua library berhasil diinstall!")

✅ Semua library berhasil diinstall!


---
## 🤖 CELL 2 — Import Library & Load Model

Di sini kita mengunduh model **Helsinki-NLP/opus-mt-id-de** dari HuggingFace.

### Tentang Model Ini
- **Nama:** `Helsinki-NLP/opus-mt-id-de`
- **Arsitektur:** MarianMT (Transformer-based)
- **Training data:** OPUS corpus (jutaan pasang kalimat Indo-Jerman)
- **Ukuran:** ~300MB (diunduh sekali, lalu cache otomatis)
- **Source:** https://huggingface.co/Helsinki-NLP/opus-mt-id-de

> ⏳ Proses download model pertama kali memerlukan waktu **1–3 menit** tergantung kecepatan internet.

In [5]:
# ============================================================
#  CELL 2 — Import Library & Load Model dari HuggingFace
# ============================================================

from transformers import MarianMTModel, MarianTokenizer
import torch
import time

# Model 1: Indonesia → Inggris
MODEL_ID_EN = "Helsinki-NLP/opus-mt-id-en"
# Model 2: Inggris → Jerman
MODEL_EN_DE = "Helsinki-NLP/opus-mt-en-de"

print("⏳ Memuat model 1: Indonesia → Inggris...")
tokenizer_id_en = MarianTokenizer.from_pretrained(MODEL_ID_EN)
model_id_en     = MarianMTModel.from_pretrained(MODEL_ID_EN)

print("⏳ Memuat model 2: Inggris → Jerman...")
tokenizer_en_de = MarianTokenizer.from_pretrained(MODEL_EN_DE)
model_en_de     = MarianMTModel.from_pretrained(MODEL_EN_DE)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id_en = model_id_en.to(device)
model_en_de = model_en_de.to(device)
model_id_en.eval()
model_en_de.eval()

print(f"\n✅ Kedua model berhasil dimuat! Berjalan di: {device.upper()}")

⏳ Memuat model 1: Indonesia → Inggris...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/796k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


✅ Kedua model berhasil dimuat! Berjalan di: CUDA


---
## ⚙️ CELL 3 — Fungsi Inti Penerjemahan

Fungsi `translate()` menerima teks Bahasa Indonesia dan mengembalikan terjemahannya dalam Bahasa Jerman.

### Parameter yang Bisa Dikustomisasi:
| Parameter | Default | Penjelasan |
|-----------|---------|------------|
| `max_length` | 512 | Panjang maksimal output token |
| `num_beams` | 4 | Jumlah beam search (lebih besar = lebih akurat, lebih lambat) |
| `early_stopping` | True | Berhenti jika semua beam sudah selesai |

In [6]:
# ============================================================
#  CELL 3 — Fungsi Inti Penerjemahan Indonesia → Jerman
# ============================================================

def _run_model(text, tokenizer, model):
    """Helper: jalankan satu model terjemahan."""
    inputs = tokenizer(
        text, return_tensors="pt",
        padding=True, truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        tokens = model.generate(
            **inputs, max_length=512,
            num_beams=4, early_stopping=True,
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(tokens[0], skip_special_tokens=True)


def translate(text: str) -> str:
    """
    Terjemahkan Indonesia → Inggris → Jerman (pipeline 2 langkah).
    """
    text = text.strip()
    if not text:
        return "Input kosong. Silakan masukkan teks terlebih dahulu."
    if len(text) > 2000:
        return "Teks terlalu panjang (maks. 2000 karakter)."

    # Langkah 1: Indonesia → Inggris
    english = _run_model(text, tokenizer_id_en, model_id_en)
    # Langkah 2: Inggris → Jerman
    german  = _run_model(english, tokenizer_en_de, model_en_de)

    return german

print("Fungsi translate() siap (pipeline: ID → EN → DE)")

Fungsi translate() siap (pipeline: ID → EN → DE)


---
## 🧪 CELL 4 — Uji Coba Cepat (Tanpa UI)

Sebelum membuka UI, kita test dulu beberapa kalimat untuk memastikan model bekerja dengan benar.

In [7]:
# ============================================================
#  CELL 4 — Uji Coba Cepat
# ============================================================

contoh_kalimat = [
    "Selamat pagi, apa kabar?",
    "Saya ingin belajar bahasa Jerman.",
    "Di mana stasiun kereta terdekat?",
    "Cuaca hari ini sangat cerah dan menyenangkan.",
    "Terima kasih atas bantuan Anda.",
]

print("=" * 60)
print("  🧪 UJI COBA PENERJEMAHAN")
print("=" * 60)

for kalimat in contoh_kalimat:
    hasil = translate(kalimat)
    print(f"\n🇮🇩 Indonesia : {kalimat}")
    print(f"🇩🇪 Jerman    : {hasil}")

print("\n" + "=" * 60)
print("✅ Uji coba selesai! Model berjalan dengan baik.")
print("=" * 60)

  🧪 UJI COBA PENERJEMAHAN

🇮🇩 Indonesia : Selamat pagi, apa kabar?
🇩🇪 Jerman    : Guten Morgen, wie geht's?

🇮🇩 Indonesia : Saya ingin belajar bahasa Jerman.
🇩🇪 Jerman    : Ich möchte Deutsch lernen.

🇮🇩 Indonesia : Di mana stasiun kereta terdekat?
🇩🇪 Jerman    : Wo ist der nächste Bahnhof?

🇮🇩 Indonesia : Cuaca hari ini sangat cerah dan menyenangkan.
🇩🇪 Jerman    : Das heutige Wetter ist sehr schön und klar.

🇮🇩 Indonesia : Terima kasih atas bantuan Anda.
🇩🇪 Jerman    : Danke für Ihre Hilfe.

✅ Uji coba selesai! Model berjalan dengan baik.


---
## 🎨 CELL 5 — Bangun Tampilan UI dengan Gradio

Di sini kita membangun antarmuka web menggunakan Gradio.

### Fitur UI yang Dibuat:
- ✍️ **Text Area** input Bahasa Indonesia
- 🔘 **Tombol Terjemahkan** dan **Tombol Reset**
- 📤 **Kotak output** hasil terjemahan Bahasa Jerman
- ⚡ **Contoh kalimat** yang bisa diklik langsung
- 📊 **Penghitung karakter** real-time

In [8]:
# ============================================================
#  CELL 5 — Bangun UI Gradio
# ============================================================

# --- CSS kustom untuk tampilan lebih menarik ---
custom_css = """
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
    max-width: 900px !important;
    margin: auto !important;
}
.title-text {
    text-align: center;
    font-size: 2rem;
    font-weight: 700;
    margin-bottom: 0.2rem;
}
.subtitle-text {
    text-align: center;
    color: #666;
    font-size: 0.95rem;
    margin-bottom: 1.5rem;
}
.translate-btn {
    background: linear-gradient(135deg, #c8102e, #ffcc00) !important;
    color: white !important;
    font-size: 1rem !important;
    font-weight: 600 !important;
    border-radius: 8px !important;
    padding: 12px !important;
}
"""

# --- Daftar contoh kalimat ---
EXAMPLES = [
    ["Selamat pagi! Apa kabar?"],
    ["Saya ingin memesan secangkir kopi."],
    ["Di mana toilet terdekat?"],
    ["Berapa harga tiket kereta ke Berlin?"],
    ["Saya tidak mengerti bahasa Jerman dengan baik."],
    ["Indonesia adalah negara kepulauan terbesar di dunia."],
    ["Tolong bantu saya menemukan jalan ke museum."],
]

# --- Fungsi untuk hitung karakter ---
def hitung_karakter(text):
    n = len(text)
    warna = "green" if n <= 1500 else ("orange" if n <= 2000 else "red")
    return f"<span style='color:{warna}'>{n}/2000 karakter</span>"

# --- Bangun UI dengan Gradio Blocks ---
with gr.Blocks(css=custom_css, title="Translator Indonesia → Jerman") as demo:

    # Header
    gr.HTML("<div class='title-text'>🇮🇩 ➡️ 🇩🇪 Translator Indonesia–Jerman</div>")
    gr.HTML("<div class='subtitle-text'>Powered by Helsinki-NLP · opus-mt-id-de · HuggingFace Transformers</div>")

    with gr.Row():
        # Kolom kiri — Input
        with gr.Column(scale=1):
            gr.Markdown("### ✍️ Teks Bahasa Indonesia")
            input_text = gr.Textbox(
                placeholder="Ketik atau tempel kalimat Bahasa Indonesia di sini...",
                lines=8,
                max_lines=15,
                label="",
                show_copy_button=True
            )
            char_count = gr.HTML("<span style='color:gray'>0/2000 karakter</span>")
            input_text.change(fn=hitung_karakter, inputs=input_text, outputs=char_count)

        # Kolom kanan — Output
        with gr.Column(scale=1):
            gr.Markdown("### 🇩🇪 Hasil Terjemahan (Deutsch)")
            output_text = gr.Textbox(
                lines=8,
                max_lines=15,
                label="",
                interactive=False,
                show_copy_button=True
            )

    # Tombol aksi
    with gr.Row():
        clear_btn     = gr.Button("🗑️ Reset", variant="secondary", scale=1)
        translate_btn = gr.Button("🔁 Terjemahkan", variant="primary", scale=3, elem_classes="translate-btn")

    # Contoh kalimat
    gr.Markdown("---\n### 💡 Contoh Kalimat (klik untuk mencoba)")
    gr.Examples(
        examples=EXAMPLES,
        inputs=input_text,
        label=""
    )

    # Info model
    with gr.Accordion("ℹ️ Tentang Model", open=False):
        gr.Markdown("""
        **Model:** `Helsinki-NLP/opus-mt-id-de`
        **Arsitektur:** MarianMT (Transformer)
        **Training Data:** OPUS Corpus (multilingual parallel corpus)
        **Bahasa:** Indonesia (id) → Jerman (de)
        **Link:** https://huggingface.co/Helsinki-NLP/opus-mt-id-de

        **Tips untuk hasil terbaik:**
        - Gunakan kalimat yang jelas dan tidak ambigu
        - Hindari singkatan atau slang yang tidak baku
        - Untuk teks panjang, coba terjemahkan per-paragraf
        """)

    # --- Event handlers ---
    translate_btn.click(
        fn=translate,
        inputs=input_text,
        outputs=output_text
    )

    input_text.submit(
        fn=translate,
        inputs=input_text,
        outputs=output_text
    )

    clear_btn.click(
        fn=lambda: ("", ""),
        inputs=None,
        outputs=[input_text, output_text]
    )

print("✅ UI Gradio berhasil dibuat. Jalankan Cell 6 untuk membuka aplikasi!")

/tmp/ipykernel_559/3386351198.py:52: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Translator Indonesia → Jerman") as demo:


✅ UI Gradio berhasil dibuat. Jalankan Cell 6 untuk membuka aplikasi!


---
## 🚀 CELL 6 — Jalankan Aplikasi!

Cell ini akan **meluncurkan aplikasi web** Gradio.

### Output yang akan muncul:
- **Local URL** → bisa dibuka jika kamu menjalankan notebook ini secara lokal
- **Public URL** → link `https://xxxxx.gradio.live` yang bisa dibuka siapapun selama 72 jam

> 🛑 Untuk **menghentikan** aplikasi, klik tombol **Stop** di toolbar Colab, atau jalankan `demo.close()` di cell baru.

In [9]:
# ============================================================
#  CELL 6 — Launch Aplikasi Gradio
# ============================================================

print("🚀 Meluncurkan aplikasi...")
print("   Tunggu beberapa detik, link akan muncul di bawah ini.\n")

demo.launch(
    share=True,         # Buat link publik (gradio.live) yang bisa dibagikan
    debug=False,        # Set True jika ingin lihat error detail
    show_error=True,    # Tampilkan pesan error di UI
    quiet=False         # Tampilkan URL
)

🚀 Meluncurkan aplikasi...
   Tunggu beberapa detik, link akan muncul di bawah ini.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6354e5e40ee50dd3c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🔧 CELL BONUS — Translate via Kode Python Langsung

Jika kamu ingin menggunakan fungsi terjemahan langsung tanpa UI (misalnya untuk keperluan batch), gunakan cell ini:

In [10]:
# ============================================================
#  CELL BONUS — Terjemahkan teks secara manual via kode
# ============================================================

# Ganti teks di bawah ini sesuai kebutuhanmu
teks_input = "Saya ingin pergi ke Jerman untuk belajar dan bekerja."

hasil = translate(teks_input)

print(f"🇮🇩 Input  : {teks_input}")
print(f"🇩🇪 Output : {hasil}")

🇮🇩 Input  : Saya ingin pergi ke Jerman untuk belajar dan bekerja.
🇩🇪 Output : Ich möchte nach Deutschland gehen, um zu studieren und zu arbeiten.


In [11]:
# ============================================================
#  CELL BONUS — Batch Translation (terjemahkan banyak kalimat)
# ============================================================

daftar_kalimat = [
    "Saya suka makan nasi goreng.",
    "Indonesia memiliki banyak pulau yang indah.",
    "Belajar bahasa asing itu menyenangkan.",
    "Tolong hubungi saya besok pagi.",
]

print("Hasil Batch Translation:")
print("-" * 60)

for i, kalimat in enumerate(daftar_kalimat, 1):
    hasil = translate(kalimat)
    print(f"{i}. 🇮🇩 {kalimat}")
    print(f"   🇩🇪 {hasil}")
    print()

Hasil Batch Translation:
------------------------------------------------------------
1. 🇮🇩 Saya suka makan nasi goreng.
   🇩🇪 Ich mag es, gebratenen Reis zu essen.

2. 🇮🇩 Indonesia memiliki banyak pulau yang indah.
   🇩🇪 Indonesien hat viele schöne Inseln.

3. 🇮🇩 Belajar bahasa asing itu menyenangkan.
   🇩🇪 Das Erlernen einer Fremdsprache macht Spaß.

4. 🇮🇩 Tolong hubungi saya besok pagi.
   🇩🇪 Bitte ruf mich morgen früh an.



---

## 📊 CELL BONUS — Evaluasi Model (BLEU Score)

Untuk mengukur kualitas terjemahan secara kuantitatif, kita bisa menggunakan metrik **BLEU Score**. Ini membandingkan terjemahan yang dihasilkan oleh model dengan terjemahan referensi yang dibuat oleh manusia.

### Langkah-langkah:
1.  **Install `sacrebleu`**: Library untuk menghitung BLEU score.
2.  **Siapkan Dataset Uji**: Contoh kalimat input, terjemahan referensi (human-made), dan terjemahan model.
3.  **Hitung BLEU Score**.

In [12]:
# ============================================================
#  Install sacrebleu
# ============================================================
!pip install sacrebleu --quiet

print("✅ Library `sacrebleu` berhasil diinstall!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
✅ Library `sacrebleu` berhasil diinstall!


In [13]:
# ============================================================
#  Siapkan Data untuk Evaluasi
# ============================================================
import sacrebleu

# Kalimat-kalimat input Bahasa Indonesia
indonesian_sentences = [
    "Halo, nama saya Budi.",
    "Saya suka makanan pedas.",
    "Apa kabar hari ini?",
    "Mari kita pergi ke pantai besok."
]

# Terjemahan referensi Bahasa Jerman (dibuat oleh manusia)
# Penting: sacrebleu mengharapkan referensi sebagai list of lists,
# di mana setiap sublist berisi satu atau lebih terjemahan referensi untuk kalimat yang sama.
# Untuk contoh ini, kita hanya punya satu referensi per kalimat.
reference_german_translations = [
    ["Hallo, mein Name ist Budi."],
    ["Ich mag scharfes Essen."],
    ["Wie geht es Ihnen heute?"],
    ["Lasst uns morgen zum Strand gehen."]
]

# Terjemahan yang dihasilkan oleh model kita
model_german_translations = []
print("⏳ Menerjemahkan kalimat-kalimat uji dengan model...")
for sentence in indonesian_sentences:
    translated_sentence = translate(sentence)
    model_german_translations.append(translated_sentence)
    print(f"  🇮🇩: {sentence}\n  🇩🇪 (Model): {translated_sentence}\n")

print("✅ Terjemahan model selesai.")

⏳ Menerjemahkan kalimat-kalimat uji dengan model...
  🇮🇩: Halo, nama saya Budi.
  🇩🇪 (Model): Hallo, mein Name ist Budi.

  🇮🇩: Saya suka makanan pedas.
  🇩🇪 (Model): Ich liebe scharfes Essen.

  🇮🇩: Apa kabar hari ini?
  🇩🇪 (Model): Wie geht's dir heute?

  🇮🇩: Mari kita pergi ke pantai besok.
  🇩🇪 (Model): Gehen wir morgen zum Strand.

✅ Terjemahan model selesai.


In [14]:
# ============================================================
#  Hitung BLEU Score
# ============================================================

# Menghitung BLEU score
# candidate: list of strings (hasil terjemahan model)
# references: list of list of strings (terjemahan referensi manusia)
bleu = sacrebleu.corpus_bleu(model_german_translations, reference_german_translations)

print("=\" * 50")
print("📊 HASIL EVALUASI BLEU SCORE")
print("=\" * 50")
print(f"BLEU Score: {bleu.score:.2f}") # Tampilkan score hingga 2 angka di belakang koma
print(f"BLEU String: {bleu.format()}") # Tampilkan detail format BLEU
print("=\" * 50")

print("\nℹ️ BLEU Score yang lebih tinggi menunjukkan kualitas terjemahan yang lebih baik.")
print("   Perlu diingat bahwa BLEU score adalah metrik statistik dan perlu diinterpretasikan ")
print("   bersama dengan evaluasi kualitatif (membaca hasil terjemahan secara langsung).")

=" * 50
📊 HASIL EVALUASI BLEU SCORE
=" * 50
BLEU Score: 100.00
BLEU String: BLEU = 100.00 100.0/100.0/100.0/100.0 (BP = 1.000 ratio = 1.000 hyp_len = 7 ref_len = 7)
=" * 50

ℹ️ BLEU Score yang lebih tinggi menunjukkan kualitas terjemahan yang lebih baik.
   Perlu diingat bahwa BLEU score adalah metrik statistik dan perlu diinterpretasikan 
   bersama dengan evaluasi kualitatif (membaca hasil terjemahan secara langsung).


---
## 📚 Referensi & Tips Lanjutan

### Troubleshooting Umum

| Masalah | Solusi |
|---------|--------|
| `ModuleNotFoundError` | Jalankan ulang Cell 1 (Install Library) |
| Model download lambat | Pastikan GPU aktif di Colab: `Runtime > Change runtime type > T4 GPU` |
| Link Gradio tidak muncul | Tunggu 30 detik, atau coba restart runtime |
| Hasil terjemahan kurang akurat | Coba kalimat yang lebih formal/baku |
| `CUDA out of memory` | Restart runtime dan jalankan ulang dari Cell 1 |

### Cara Aktifkan GPU di Google Colab
1. Klik menu **Runtime** di toolbar atas
2. Pilih **Change runtime type**
3. Di bagian *Hardware accelerator*, pilih **T4 GPU**
4. Klik **Save**, lalu jalankan ulang semua cell

### Ingin Kembangkan Lebih Lanjut?
- 🔄 **Tambah bahasa lain:** Ganti `opus-mt-id-de` dengan model lain, misal `opus-mt-id-en` (Indo→Inggris)
- 📄 **Upload file:** Tambahkan `gr.File()` di UI untuk terjemahkan file `.txt`
- 💾 **Simpan hasil:** Tambahkan tombol download hasil terjemahan
- 🌐 **Deploy permanen:** Upload ke HuggingFace Spaces untuk hosting gratis